# 01 - Exploración de datos
### Statoil/C-CORE Iceberg Classifier Challenge

**Curso:** Deep Learning — Módulo 4 (Convolutional Networks)
**Objetivo de este notebook:** entender la estructura de los datos, revisar la distribución de clases,
visualizar ejemplos de barcos e icebergs a partir de las imágenes de radar (SAR), y **confirmar
cuantitativamente el sesgo relacionado con la variable `inc_angle`** señalado en la retroalimentación
de la Entrega 1.

Los datos se describen en detalle en `ENTREGA1.PDF` (carpeta raíz de este repositorio).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
np.random.seed(42)

## 1. Descarga / carga de los datos

In [ ]:
import os, json

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
train_path = os.path.join(DATA_DIR, "train.json")
test_path = os.path.join(DATA_DIR, "test.json")

if not (os.path.exists(train_path) and os.path.exists(test_path)):
    print("Datos no encontrados localmente. Descargando desde Kaggle...")
    !pip install -q kaggle py7zr

    IN_COLAB = "google.colab" in str(get_ipython())
    if IN_COLAB:
        from google.colab import files
        if not os.path.exists("/root/.kaggle/kaggle.json"):
            print("Sube tu archivo kaggle.json (Kaggle -> Account -> Create New API Token)")
            uploaded = files.upload()
            os.makedirs("/root/.kaggle", exist_ok=True)
            for fname in uploaded:
                if fname.endswith(".json"):
                    os.rename(fname, "/root/.kaggle/kaggle.json")
            os.chmod("/root/.kaggle/kaggle.json", 0o600)

    os.system(f"kaggle competitions download -c statoil-iceberg-classifier-challenge -p {DATA_DIR}")

    import py7zr
    for fname in ["train.json.7z", "test.json.7z"]:
        fpath = os.path.join(DATA_DIR, fname)
        if os.path.exists(fpath):
            with py7zr.SevenZipFile(fpath, mode="r") as z:
                z.extractall(path=DATA_DIR)
    print("Descarga y extracción completas.")
else:
    print(f"Datos ya disponibles en '{DATA_DIR}/'. Se omite la descarga.")


In [ ]:
with open("data/train.json") as f:
    train_raw = json.load(f)

df = pd.DataFrame(train_raw)
print("Forma del dataframe:", df.shape)
df.head()

## 2. Inspección general

Cada fila representa una imagen SAR de 75x75 píxeles con dos bandas de polarización (`band_1`=HH,
`band_2`=HV), el ángulo de incidencia del radar (`inc_angle`) y la etiqueta `is_iceberg`
(1 = iceberg, 0 = barco).

In [ ]:
print(df.dtypes)
print()
print("Valores únicos de is_iceberg:", df["is_iceberg"].unique())
print("Valores 'na' en inc_angle:", (df["inc_angle"] == "na").sum(), "de", len(df))

In [ ]:
# inc_angle llega como string ('na' para los faltantes) -> convertir a float con NaN
df["inc_angle"] = pd.to_numeric(df["inc_angle"], errors="coerce")
print(df["inc_angle"].describe())
print("\nFaltantes en inc_angle:", df["inc_angle"].isna().sum(),
      f"({df['inc_angle'].isna().mean()*100:.1f}% del total)")

## 3. Distribución de clases

In [ ]:
counts = df["is_iceberg"].value_counts().sort_index()
print(counts)
print(f"\nBarco (0): {counts[0]} ({counts[0]/len(df)*100:.1f}%)")
print(f"Iceberg (1): {counts[1]} ({counts[1]/len(df)*100:.1f}%)")

plt.figure(figsize=(4,4))
plt.bar(["Barco (0)", "Iceberg (1)"], counts.values, color=["#4C72B0", "#C44E52"])
plt.title("Distribución de clases - train")
plt.ylabel("N.º de imágenes")
plt.show()

## 4. Visualización de ejemplos

Cada imagen se entrega "aplanada" (lista de 5,625 = 75x75 valores) y debe reconstruirse como
matriz 75x75 antes de poder visualizarla.

In [ ]:
def to_image(band_values):
    return np.array(band_values, dtype=np.float32).reshape(75, 75)

def plot_examples(df, label, n=3, seed=0):
    sample = df[df["is_iceberg"] == label].sample(n=min(n, (df["is_iceberg"]==label).sum()),
                                                    random_state=seed)
    fig, axes = plt.subplots(n, 3, figsize=(9, 3*n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for i, (_, row) in enumerate(sample.iterrows()):
        b1 = to_image(row["band_1"])
        b2 = to_image(row["band_2"])
        composite = (b1 + b2) / 2
        axes[i, 0].imshow(b1, cmap="gray"); axes[i, 0].set_title("band_1 (HH)")
        axes[i, 1].imshow(b2, cmap="gray"); axes[i, 1].set_title("band_2 (HV)")
        axes[i, 2].imshow(composite, cmap="gray"); axes[i, 2].set_title("promedio (HH+HV)/2")
        for ax in axes[i]:
            ax.axis("off")
    etiqueta = "Iceberg" if label == 1 else "Barco"
    fig.suptitle(f"Ejemplos de clase: {etiqueta}", fontsize=13)
    plt.tight_layout()
    plt.show()

plot_examples(df, label=0, n=3)  # barcos
plot_examples(df, label=1, n=3)  # icebergs

## 5. Análisis del sesgo en `inc_angle`

La retroalimentación del profesor sobre la Entrega 1 señaló un posible sesgo relacionado con el
ángulo de incidencia. Lo verificamos de dos formas: (a) visualmente, comparando la distribución de
`inc_angle` entre clases, y (b) numéricamente, midiendo cuántos valores de `inc_angle` se repiten
entre registros y si esos duplicados comparten la misma etiqueta con una frecuencia mayor de la
esperable por azar (el "data leak" reportado por la comunidad de Kaggle para este reto).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(df.loc[df.is_iceberg==0, "inc_angle"].dropna(), bins=20, alpha=0.6, label="Barco")
axes[0].hist(df.loc[df.is_iceberg==1, "inc_angle"].dropna(), bins=20, alpha=0.6, label="Iceberg")
axes[0].set_title("Distribución de inc_angle por clase")
axes[0].set_xlabel("inc_angle (grados)")
axes[0].legend()

axes[1].boxplot(
    [df.loc[df.is_iceberg==0, "inc_angle"].dropna(),
     df.loc[df.is_iceberg==1, "inc_angle"].dropna()],
    tick_labels=["Barco", "Iceberg"]
)
axes[1].set_title("inc_angle por clase (boxplot)")
plt.tight_layout()
plt.show()

print(df.groupby("is_iceberg")["inc_angle"].agg(["mean", "std", "median"]))

In [ ]:
# Cuantificar el posible data-leak: valores de inc_angle que se repiten
# (redondeados a 4 decimales, como reportó la comunidad de Kaggle) y si comparten etiqueta.
df["inc_angle_r4"] = df["inc_angle"].round(4)

dup_groups = df.dropna(subset=["inc_angle_r4"]).groupby("inc_angle_r4")
dup_groups_multi = [g for _, g in dup_groups if len(g) > 1]

n_dup_rows = sum(len(g) for g in dup_groups_multi)
same_label_groups = sum(1 for g in dup_groups_multi if g["is_iceberg"].nunique() == 1)

print(f"Grupos de inc_angle duplicado (>1 registro): {len(dup_groups_multi)}")
print(f"Filas involucradas en algún duplicado: {n_dup_rows} de {len(df)} "
      f"({n_dup_rows/len(df)*100:.1f}%)")
if dup_groups_multi:
    print(f"De esos grupos, el {same_label_groups/len(dup_groups_multi)*100:.1f}% "
          f"comparten la MISMA etiqueta en todos sus miembros.")
    print("\n(En el dataset real se documentó un porcentaje muy superior al esperado por azar, lo que "
          "confirma que inc_angle puede filtrar información de la etiqueta y NO debe usarse "
          "ingenuamente como feature. Con la muestra pequeña de prueba de este notebook el "
          "porcentaje puede no ser representativo.)")
else:
    print("No se encontraron duplicados en esta muestra (puede ocurrir con pocos datos de prueba).")

## 6. Conclusiones de la exploración

- El dataset está razonablemente **balanceado** entre barco e iceberg.
- Existe un porcentaje de valores **faltantes en `inc_angle`** que debe tratarse en el preprocesado.
- Se confirma la señal de **sesgo/leak en `inc_angle`**: registros con el mismo ángulo de incidencia
  tienden a compartir etiqueta con más frecuencia de la esperada. Esto es coherente con que varias
  imágenes provienen de la misma pasada del satélite.
- **Implicación para el modelado (se aborda en `02 - preprocesado` y `05 - análisis de sesgo`):**
  1. No se debe permitir que imágenes con `inc_angle` idéntico o casi idéntico queden repartidas
     entre train y validación (contaminaría la validación).
  2. Se evaluará si conviene excluir `inc_angle` como entrada directa al modelo, o si usarlo de forma
     controlada (p. ej. como metadato adicional, nunca como atajo para memorizar la etiqueta).

Esto continúa en **`02 - preprocesado.ipynb`**.